In [2]:
import tensorflow as tf
keras = tf.keras
from keras import layers, Model
from keras.applications import MobileNetV2
from keras.applications.mobilenet_v2 import preprocess_input
from keras.utils import image_dataset_from_directory

In [4]:
IMG_SIZE = (160, 160)
BATCH_SIZE = 32

train_ds = image_dataset_from_directory(
    "dataset/train",
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    label_mode="binary",
    shuffle=True
)

val_ds = image_dataset_from_directory(
    "dataset/validation",
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    label_mode="binary",
    shuffle=False
)

Found 20000 files belonging to 2 classes.


2026-10-29 15:57:03.297300: I tensorflow/core/platform/cpu_feature_guard.cc:193] This TensorFlow binary is optimized with oneAPI Deep Neural Network Library (oneDNN) to use the following CPU instructions in performance-critical operations:  AVX2 AVX_VNNI FMA
To enable them in other operations, rebuild TensorFlow with the appropriate compiler flags.
2026-10-29 15:57:03.301101: I tensorflow/c/logging.cc:34] DirectML: creating device on adapter 0 (Intel(R) Iris(R) Xe Graphics)
2026-10-29 15:57:03.426139: I tensorflow/core/common_runtime/pluggable_device/pluggable_device_factory.cc:306] Could not identify NUMA node of platform GPU ID 0, defaulting to 0. Your kernel may not have been built with NUMA support.
2026-10-29 15:57:03.426479: W tensorflow/core/common_runtime/pluggable_device/pluggable_device_bfc_allocator.cc:28] Overriding allow_growth setting because force_memory_growth was requested by the device.
2026-10-29 15:57:03.426776: I tensorflow/core/common_runtime/pluggable_device/plug

Found 5000 files belonging to 2 classes.


In [6]:
train_ds = train_ds.map(
    lambda images, labels: (preprocess_input(images), labels)
)

val_ds = val_ds.map(
    lambda images, labels: (preprocess_input(images), labels)
)

In [7]:
base_model = MobileNetV2(
    weights="imagenet",
    include_top=False,
    input_shape=(160, 160, 3)
)

base_model.trainable = False

In [8]:
inputs = layers.Input(shape=(160, 160, 3))

x = base_model(inputs, training=False)
x = layers.GlobalAvgPool2D()(x)
x = layers.Dropout(0.2)(x)

outputs = layers.Dense(1, activation="sigmoid")(x)

model = Model(inputs, outputs)
model.summary()

Model: "model"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 input_2 (InputLayer)        [(None, 160, 160, 3)]     0         
                                                                 
 mobilenetv2_1.00_160 (Funct  (None, 5, 5, 1280)       2257984   
 ional)                                                          
                                                                 
 global_average_pooling2d (G  (None, 1280)             0         
 lobalAveragePooling2D)                                          
                                                                 
 dropout (Dropout)           (None, 1280)              0         
                                                                 
 dense (Dense)               (None, 1)                 1281      
                                                                 
Total params: 2,259,265
Trainable params: 1,281
Non-trainable

In [9]:
model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=1e-3),
    loss="binary_crossentropy",
    metrics=["accuracy"]
)

In [10]:
callbacks = [
    keras.callbacks.EarlyStopping(
        monitor="val_loss",
        patience=2,
        restore_best_weights=True
    )
]

history = model.fit(
    train_ds, 
    validation_data=val_ds,
    epochs=5,
    callbacks=callbacks
)

val_loss, val_accuracy = model.evaluate(val_ds)

print("Validation loss:", val_loss)
print("Validation accuracy:", val_accuracy)

Epoch 1/5


2026-10-29 16:49:19.051550: I tensorflow/core/grappler/optimizers/custom_graph_optimizer_registry.cc:114] Plugin optimizer for device_type GPU is enabled.
2026-10-29 16:49:19.473689: I tensorflow/core/common_runtime/pluggable_device/pluggable_device_factory.cc:306] Could not identify NUMA node of platform GPU ID 0, defaulting to 0. Your kernel may not have been built with NUMA support.
2026-10-29 16:49:19.473873: I tensorflow/core/common_runtime/pluggable_device/pluggable_device_factory.cc:272] Created TensorFlow device (/job:localhost/replica:0/task:0/device:GPU:0 with 6941 MB memory) -> physical PluggableDevice (device: 0, name: DML, pci bus id: <undefined>)


625/625 [==============================] - ETA: 0s - loss: 0.6861 - accuracy: 0.5671

2026-10-29 16:52:03.964901: I tensorflow/core/grappler/optimizers/custom_graph_optimizer_registry.cc:114] Plugin optimizer for device_type GPU is enabled.
2026-10-29 16:52:04.329794: I tensorflow/core/common_runtime/pluggable_device/pluggable_device_factory.cc:306] Could not identify NUMA node of platform GPU ID 0, defaulting to 0. Your kernel may not have been built with NUMA support.
2026-10-29 16:52:04.330031: I tensorflow/core/common_runtime/pluggable_device/pluggable_device_factory.cc:272] Created TensorFlow device (/job:localhost/replica:0/task:0/device:GPU:0 with 6941 MB memory) -> physical PluggableDevice (device: 0, name: DML, pci bus id: <undefined>)


625/625 [==============================] - 225s 324ms/step - loss: 0.6861 - accuracy: 0.5671 - val_loss: 1.0024 - val_accuracy: 0.5574
Epoch 2/5
625/625 [==============================] - 171s 273ms/step - loss: 0.6572 - accuracy: 0.6118 - val_loss: 1.1136 - val_accuracy: 0.5642
Epoch 3/5
157/157 [==============================] - 34s 213ms/step - loss: 1.0024 - accuracy: 0.5574
Validation loss: 1.0024254322052002
Validation accuracy: 0.5574000477790833


In [13]:
for images, labels in val_ds.take(1):
    probabilities = model.predict(images, verbose=0).flatten()

    for i in range(min(5, len(probabilities))):
        predicted_index = int(probabilities[i] <= 0.5)
        # predicted_class = train_ds.class_names[predicted_index]
        # actual_class = train_ds.class_names[int(labels[i].numpy())]
        actual_index = int(labels[i])

        print(
            # f"Predicted: {predicted_class}, "
            f"Predicted: {predicted_index}, "
            # f"Actual: {actual_class}, "
            f"Actual: {actual_index}, "
            f"Score: {probabilities[i]:.3f}"
        )

Predicted: 0, Actual: 0, Score: 0.661
Predicted: 1, Actual: 0, Score: 0.347
Predicted: 0, Actual: 0, Score: 0.901
Predicted: 0, Actual: 0, Score: 0.778
Predicted: 0, Actual: 0, Score: 0.995
